In [1]:
"""
 Dataset: "Trending YouTube Video Statistics" by Mitchell J (datasnaek)
         https://www.kaggle.com/datasets/datasnaek/youtube-new
 
Setup:
  1. Download the dataset from Kaggle
  2. Place all CSV files and JSON files in the DATA_DIR folder
  3. pip install pandas numpy scikit-learn matplotlib seaborn vaderSentiment
  4. Run: python viral_prediction_pipeline.py
 
Dataset structure:
  data/
    USvideos.csv
    GBvideos.csv
    CAvideos.csv
    DEvideos.csv
    FRvideos.csv
    ... (other country CSVs)
    US_category_id.json
    GB_category_id.json
    ... (other country JSONs)
 
CSV columns:
  video_id, trending_date, title, channel_title, category_id,
  publish_time, tags, views, likes, dislikes, comment_count,
  thumbnail_link, comments_disabled, ratings_disabled,
  video_error_or_removed, description
"""
import os
import sys
import json
import time
import warnings
from datetime import datetime
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    RandomizedSearchCV,
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, f1_score, roc_auc_score,
    accuracy_score, precision_score, recall_score, make_scorer,
)
from sklearn.inspection import permutation_importance
 
warnings.filterwarnings('ignore')
 
 
# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    # Path to the folder containing the Kaggle CSV and JSON files
    DATA_DIR = './Dataset/'
 
    # Which country files to load (set to None to load all available)
    COUNTRIES = ['US', 'GB', 'CA']  # Change or set to None for all
 
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    CV_FOLDS = 5
    HYPERPARAM_ITER = 20
 
    # Virality: top X percentile of engagement within each category
    VIRALITY_PERCENTILE = 85  # Top 15% = viral
 
    # Output
    OUTPUT_DIR = './output/'
    FIGURE_DIR = './output/figures/'
    FIGURE_DPI = 180
 
    # Plotting
    COLORS = ['#4F6D7A', '#C06C84', '#F67280', '#F8B500', '#355C7D', '#6C5B7B']
 

##  Helper Functions
Utility functions used throughout the pipeline:
- `setup()` creates output directories and configures Matplotlib defaults
- `save_fig()` saves figures as PNG files and closes them to free memory
- `header()` and `subheader()` print formatted stage titles to track pipeline progress

In [2]:
# =============================================================================
# HELPERS
# =============================================================================
def setup():
    os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
    os.makedirs(Config.FIGURE_DIR, exist_ok=True)
    plt.rcParams.update({
        'font.family': 'sans-serif', 'font.size': 11,
        'axes.titlesize': 13, 'axes.labelsize': 11,
        'figure.dpi': Config.FIGURE_DPI, 'savefig.dpi': Config.FIGURE_DPI,
        'savefig.bbox': 'tight', 'savefig.pad_inches': 0.15,
    })
 
 
def save_fig(fig, name):
    path = os.path.join(Config.FIGURE_DIR, f'{name}.png')
    fig.savefig(path)
    plt.close(fig)
    print(f"  Saved: {path}")
 
 
def header(text):
    print(f"\n{'='*70}\n  {text}\n{'='*70}")
 
 
def subheader(text):
    print(f"\n--- {text} ---")
 
 

## Stage 1 — Data Loading and Preprocessing
The core data pipeline. This cell defines two functions:

**`load_category_mapping()`** reads JSON files that map numeric category IDs (e.g. 24) to human-readable names (e.g. "Entertainment").

**`load_data()`** performs the full preprocessing pipeline:
1. Loads CSV files for each country with UTF-8/latin-1 fallback encoding
2. Parses the unusual `YY.DD.MM` date format in the `trending_date` column
3. Fixes a timezone mismatch: `publish_time` is timezone-aware but `trending_date` is not, so `tz_localize(None)` strips the timezone before computing `days_to_trending`
4. Deduplicates by keeping only the latest trending observation per video per country (videos can trend for multiple consecutive days)
5. Computes `channel_median_views` as a proxy for subscriber count (which the dataset lacks)
6. Defines virality using a normalised engagement score: `(views + likes + comments) / channel_median_views`, with the top 15th percentile within each category labelled viral

In [3]:
# =============================================================================
# STAGE 1: LOAD AND PREPROCESS THE KAGGLE DATASET
# =============================================================================
def load_category_mapping(data_dir, country_code):
    """
    Load category names from the JSON file for a given country.
    Returns a dict mapping category_id (int) to category name (str).
    """
    json_path = os.path.join(data_dir, f'{country_code}_category_id.json')
    if not os.path.exists(json_path):
        return {}
 
    with open(json_path, 'r') as f:
        data = json.load(f)
 
    mapping = {}
    for item in data['items']:
        cat_id = int(item['id'])
        cat_name = item['snippet']['title']
        mapping[cat_id] = cat_name
 
    return mapping
 
 
def load_data():
    """
    Load and preprocess the Kaggle "Trending YouTube Video Statistics" dataset.
    Handles multiple country files, category JSON mapping, deduplication,
    and all necessary cleaning.
    """
    header("STAGE 1: DATA LOADING AND PREPROCESSING")
 
    data_dir = Config.DATA_DIR
 
    # Find available CSV files
    all_csvs = [f for f in os.listdir(data_dir) if f.endswith('videos.csv')]
    if not all_csvs:
        print(f"ERROR: No CSV files found in {data_dir}")
        print(f"Expected files like USvideos.csv, GBvideos.csv, etc.")
        sys.exit(1)
 
    # Filter to requested countries if specified
    if Config.COUNTRIES:
        csvs = [f for f in all_csvs
                if f[:2] in Config.COUNTRIES]
    else:
        csvs = all_csvs
 
    print(f"  Found {len(csvs)} CSV files: {csvs}")
 
    # Load and combine all country files
    dfs = []
    for csv_file in csvs:
        country = csv_file[:2]
        filepath = os.path.join(data_dir, csv_file)
 
        try:
            df = pd.read_csv(filepath, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(filepath, encoding='latin-1')
 
        df['country'] = country
 
        # Load category mapping for this country
        cat_map = load_category_mapping(data_dir, country)
        if cat_map:
            df['category'] = df['category_id'].map(cat_map).fillna('Other')
        else:
            df['category'] = 'Unknown'
 
        dfs.append(df)
        print(f"  Loaded {csv_file}: {len(df)} rows, {df['category'].nunique()} categories")
 
    df = pd.concat(dfs, ignore_index=True)
    print(f"\n  Combined: {len(df)} rows")
 
    # --- Parse dates ---
    # trending_date format is YY.DD.MM in this dataset
    df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m', errors='coerce')
 
    # publish_time is ISO format
    df['publish_time'] = pd.to_datetime(df['publish_time'], errors='coerce')
 
    # --- Deduplication ---
    # Videos appear on trending for multiple days; keep the latest observation
    df = df.sort_values('trending_date')
    df = df.drop_duplicates(subset=['video_id', 'country'], keep='last')
    print(f"  After deduplication: {len(df)} rows")
 
    # --- Remove problematic rows ---
    df = df[df['video_error_or_removed'] == False]
    df = df[df['views'] > 0]
    df = df.dropna(subset=['title', 'publish_time'])
    print(f"  After cleaning: {len(df)} rows")
 
    # --- Compute metadata features ---
    subheader("Computing metadata features")
 
    # Title features
    df['title_length'] = df['title'].str.len().clip(1, 200)
    df['title_word_count'] = df['title'].str.split().str.len().clip(1, 50)
 
    # Tag features
    # Tags are pipe-separated: "tag1|tag2|tag3" or "[none]" if empty
    df['tags'] = df['tags'].fillna('[none]')
    df['tag_count'] = df['tags'].apply(
        lambda x: len(x.split('|')) if x != '[none]' else 0
    )
 
    # Description
    df['description'] = df['description'].fillna('')
    df['has_description'] = (df['description'].str.len() > 0).astype(int)
 
    # Temporal features
    df['publish_hour'] = df['publish_time'].dt.hour
    df['publish_day'] = df['publish_time'].dt.dayofweek
 
    # Time to trending (days between publish and first trending)
    df['publish_time_naive'] = df['publish_time'].dt.tz_localize(None)
    df['days_to_trending'] = (df['trending_date'] - df['publish_time_naive']).dt.days
    df['days_to_trending'] = df['days_to_trending'].clip(0, 365).fillna(0)
    df = df.drop(columns=['publish_time_naive'])
    df['days_to_trending'] = df['days_to_trending'].clip(0, 365).fillna(0)
 
    # Disabled features (boolean to int)
    df['comments_disabled'] = df['comments_disabled'].astype(int)
    df['ratings_disabled'] = df['ratings_disabled'].astype(int)
 
    # --- Sentiment analysis ---
    subheader("Computing title sentiment")
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        analyzer = SentimentIntensityAnalyzer()
        df['title_sentiment'] = df['title'].apply(
            lambda x: analyzer.polarity_scores(str(x))['compound']
        )
        print("  Using VADER sentiment analyzer")
    except ImportError:
        # Heuristic fallback: exclamation marks, caps ratio, question marks
        df['title_sentiment'] = (
            df['title'].str.count('!') * 0.15 +
            df['title'].str.count(r'\?') * 0.05 +
            df['title'].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)) * 0.3
            - 0.1
        ).clip(-1, 1)
        print("  Using heuristic sentiment (install vaderSentiment for better results)")
 
    # --- Engagement ratios (Tier 2 features) ---
    df['likes_to_views'] = df['likes'] / df['views'].clip(1)
    df['dislikes_to_views'] = df['dislikes'] / df['views'].clip(1)
    df['comments_to_views'] = df['comment_count'] / df['views'].clip(1)
    df['like_dislike_ratio'] = df['likes'] / (df['likes'] + df['dislikes']).clip(1)
 
    # --- Channel proxy for subscriber count ---
    # This dataset doesn't have subscriber counts, so we use median views
    # per channel as a rough proxy for channel size
    channel_median = df.groupby('channel_title')['views'].median()
    df['channel_median_views'] = df['channel_title'].map(channel_median)
 
    # --- Engagement score (normalised by channel size) ---
    # Total engagement relative to what's typical for this channel
    df['engagement_score'] = (
        (df['views'] + df['likes'] + df['comment_count'])
        / df['channel_median_views'].clip(1)
    )
 
    # --- Virality label ---
    # A video is viral if its engagement score is in the top percentile
    # within its category. This normalises across categories.
    subheader("Computing virality labels")
    df['viral'] = 0
    for cat in df['category'].unique():
        mask = df['category'] == cat
        threshold = df.loc[mask, 'engagement_score'].quantile(
            Config.VIRALITY_PERCENTILE / 100
        )
        df.loc[mask & (df['engagement_score'] >= threshold), 'viral'] = 1
 
    n_viral = df['viral'].sum()
    print(f"  Viral videos: {n_viral} ({n_viral/len(df)*100:.1f}%)")
    print(f"  Non-viral:    {len(df) - n_viral} ({(len(df)-n_viral)/len(df)*100:.1f}%)")
    print(f"  Categories:   {df['category'].nunique()}")
 
    # Save processed data
    df.to_csv(os.path.join(Config.OUTPUT_DIR, 'processed_data.csv'), index=False)
    print(f"\n  Saved processed data to {Config.OUTPUT_DIR}processed_data.csv")
 
    return df

##  Stage 2 — Exploratory Data Analysis
Generates five visualisations that directly inform the modelling decisions:

- Figure 1 (Class Distribution)
- Figure 2 (Categories)
- Figure 3 (Correlation Matrix)
- Figure 4 (Temporal Patterns)
- Figure 5 (Engagement Distributions)

In [4]:
# =============================================================================
# STAGE 2: EXPLORATORY DATA ANALYSIS
# =============================================================================
def run_eda(df):
    header("STAGE 2: EXPLORATORY DATA ANALYSIS")
 
    # --- Figure 1: Class distribution ---
    subheader("Figure 1: Class Distribution")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
 
    log_score = np.log10(df['engagement_score'].clip(1e-6))
    axes[0].hist(log_score, bins=60, color=Config.COLORS[0], alpha=0.8, edgecolor='white')
    pct = np.log10(df['engagement_score'].quantile(Config.VIRALITY_PERCENTILE / 100))
    axes[0].axvline(pct, color=Config.COLORS[1], ls='--', lw=2,
                    label=f'{Config.VIRALITY_PERCENTILE}th percentile')
    axes[0].set_xlabel('log10(Engagement Score)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Engagement Score')
    axes[0].legend(fontsize=9)
 
    counts = df['viral'].value_counts().sort_index()
    axes[1].bar(['Not Viral', 'Viral'], counts.values,
                color=[Config.COLORS[0], Config.COLORS[1]], edgecolor='white')
    for i, v in enumerate(counts.values):
        axes[1].text(i, v + max(counts)*0.02, f'{v} ({v/len(df)*100:.1f}%)',
                     ha='center', fontsize=10, fontweight='bold')
    axes[1].set_title('Class Distribution')
    axes[1].set_ylabel('Count')
    plt.tight_layout()
    save_fig(fig, '01_class_distribution')
 
    # --- Figure 2: Category analysis ---
    subheader("Figure 2: Category Analysis")
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
 
    cat_stats = df.groupby('category')['viral'].agg(['sum', 'count'])
    cat_stats['rate'] = cat_stats['sum'] / cat_stats['count'] * 100
 
    cat_by_count = cat_stats.sort_values('count', ascending=True)
    axes[0].barh(cat_by_count.index, cat_by_count['count'],
                 color=Config.COLORS[0], alpha=0.8)
    axes[0].set_xlabel('Number of Videos')
    axes[0].set_title('Videos per Category')
 
    cat_by_rate = cat_stats.sort_values('rate', ascending=True)
    axes[1].barh(cat_by_rate.index, cat_by_rate['rate'],
                 color=Config.COLORS[1], alpha=0.8)
    axes[1].set_xlabel('Viral Rate (%)')
    axes[1].set_title('Viral Rate by Category')
    plt.tight_layout()
    save_fig(fig, '02_categories')
 
    # --- Figure 3: Correlation heatmap ---
    subheader("Figure 3: Correlation Matrix")
    num_cols = ['views', 'likes', 'dislikes', 'comment_count',
                'title_length', 'title_word_count', 'tag_count',
                'title_sentiment', 'publish_hour',
                'likes_to_views', 'comments_to_views', 'like_dislike_ratio']
    available = [c for c in num_cols if c in df.columns]
 
    fig, ax = plt.subplots(figsize=(9, 7))
    corr = df[available].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, square=True, linewidths=0.5, ax=ax,
                annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
    ax.set_title('Feature Correlation Matrix', pad=15)
    plt.tight_layout()
    save_fig(fig, '03_correlation')
 
    # --- Figure 4: Temporal patterns ---
    subheader("Figure 4: Temporal Patterns")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
 
    hour_viral = df.groupby('publish_hour')['viral'].mean() * 100
    axes[0].bar(hour_viral.index, hour_viral.values,
                color=Config.COLORS[0], alpha=0.8, edgecolor='white')
    axes[0].set_xlabel('Publish Hour (UTC)')
    axes[0].set_ylabel('Viral Rate (%)')
    axes[0].set_title('Viral Rate by Publish Hour')
    axes[0].set_xticks(range(0, 24, 3))
 
    day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    day_viral = df.groupby('publish_day')['viral'].mean() * 100
    axes[1].bar(day_labels, day_viral.values,
                color=Config.COLORS[4], alpha=0.8, edgecolor='white')
    axes[1].set_xlabel('Publish Day')
    axes[1].set_ylabel('Viral Rate (%)')
    axes[1].set_title('Viral Rate by Day of Week')
    plt.tight_layout()
    save_fig(fig, '04_temporal')
 
    # --- Figure 5: Engagement by viral status ---
    subheader("Figure 5: Engagement Distributions")
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    metrics = ['likes_to_views', 'comments_to_views', 'title_sentiment']
    titles = ['Likes to Views Ratio', 'Comments to Views Ratio', 'Title Sentiment']
 
    for i, (metric, title) in enumerate(zip(metrics, titles)):
        for label, color in [(0, Config.COLORS[0]), (1, Config.COLORS[1])]:
            subset = df[df['viral'] == label][metric].dropna()
            axes[i].hist(subset, bins=40, alpha=0.6, color=color,
                         label='Viral' if label == 1 else 'Not Viral',
                         density=True, edgecolor='white')
        axes[i].set_xlabel(metric)
        axes[i].set_ylabel('Density')
        axes[i].set_title(title)
        axes[i].legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, '05_engagement_distributions')
 
    # --- Print summary ---
    subheader("Summary Statistics")
    print(f"  Total videos:     {len(df)}")
    print(f"  Viral videos:     {df['viral'].sum()} ({df['viral'].mean()*100:.1f}%)")
    print(f"  Categories:       {df['category'].nunique()}")
    print(f"  Countries:        {df['country'].nunique()}")
    print(f"  Median views:     {df['views'].median():,.0f}")
    print(f"  Median likes:     {df['likes'].median():,.0f}")
    print(f"  Median comments:  {df['comment_count'].median():,.0f}")
 
    return df

## Stage 3 — Feature Engineering
Transforms raw data into model-ready features using three key techniques:

- **Cyclical encoding:** Converts publish hour and day of week into sine/cosine pairs so the model understands that 11pm and midnight are close together (not 23 apart)
- **Log transformation:** Reduces extreme skewness in views (from 24.99 to -0.06) and channel median views
- **Label encoding:** Converts 18 category names into numeric values

Features are organised into two tiers for controlled comparison:
- **Tier 1 (14 features):** Metadata only — title length, word count, tag count, VADER sentiment, cyclical time, channel size, days to trending, category, disabled flags
- **Tier 2 (18 features):** Adds 4 engagement ratios — likes/views, comments/views, like-dislike ratio, dislikes/views

In [5]:
# =============================================================================
# STAGE 3: FEATURE ENGINEERING
# =============================================================================
def engineer_features(df):
    header("STAGE 3: FEATURE ENGINEERING")
 
    # Cyclical encoding for time
    df['hour_sin'] = np.sin(2 * np.pi * df['publish_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['publish_hour'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['publish_day'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['publish_day'] / 7)
    print("  Created cyclical time features")
 
    # Log transforms for skewed features
    df['log_views'] = np.log1p(df['views'])
    df['log_channel_median'] = np.log1p(df['channel_median_views'])
    print(f"  views skew: {df['views'].skew():.2f} -> {df['log_views'].skew():.2f}")
 
    # Category encoding
    le = LabelEncoder()
    df['category_encoded'] = le.fit_transform(df['category'])
    print(f"  Encoded {len(le.classes_)} categories")
 
    # Define feature tiers
    tier1_features = [
        'title_length', 'title_word_count', 'tag_count', 'title_sentiment',
        'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
        'has_description', 'comments_disabled', 'ratings_disabled',
        'log_channel_median', 'category_encoded', 'days_to_trending',
    ]
 
    tier2_features = tier1_features + [
        'likes_to_views', 'comments_to_views', 'like_dislike_ratio',
        'dislikes_to_views',
    ]
 
    # Check all features exist
    for feat in tier2_features:
        if feat not in df.columns:
            print(f"  WARNING: {feat} not found in dataframe")
 
    print(f"\n  Tier 1 features (metadata only): {len(tier1_features)}")
    print(f"  Tier 2 features (+ engagement):  {len(tier2_features)}")
 
    return df, tier1_features, tier2_features

## Stage 4 — SMOTE Oversampling
Handles the class imbalance problem (85% non-viral, 15% viral). Without SMOTE, models default to predicting "not viral" for everything and achieve 85% accuracy while being useless.

In [6]:
# =============================================================================
# STAGE 4: SMOTE
# =============================================================================
def apply_smote(X, y, random_state=42):
    """Apply SMOTE oversampling. Uses imbalanced-learn if available, else manual."""
    try:
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=random_state)
        X_res, y_res = sm.fit_resample(X, y)
        print(f"  SMOTE (imbalanced-learn): {len(y)} -> {len(y_res)}")
        return X_res, y_res
    except ImportError:
        pass
 
    rng = np.random.RandomState(random_state)
    min_idx = np.where(y == 1)[0]
    maj_idx = np.where(y == 0)[0]
    n_gen = len(maj_idx) - len(min_idx)
 
    if n_gen <= 0:
        return X, y
 
    X_min = X[min_idx]
    synthetic = []
    for _ in range(n_gen):
        i = rng.randint(0, len(X_min))
        j = rng.randint(0, len(X_min))
        while j == i and len(X_min) > 1:
            j = rng.randint(0, len(X_min))
        lam = rng.random()
        synthetic.append(X_min[i] + lam * (X_min[j] - X_min[i]))
 
    X_res = np.vstack([X, np.array(synthetic)])
    y_res = np.hstack([y, np.ones(n_gen)])
    print(f"  SMOTE (manual): {len(y)} -> {len(y_res)}")
    return X_res, y_res
 

## Stage 5 — Model Building and Training
**`build_models()`** defines four classifiers spanning a complexity spectrum, each paired with its hyperparameter search space:
- **Logistic Regression:** Linear baseline, searches over regularisation strength C
- **Random Forest:** Bagging ensemble, searches over number of trees, max depth, and leaf settings
- **Gradient Boosting:** Sequential boosting, searches over estimators, depth, learning rate, and subsample ratio
- **Neural Network:** MLPClassifier with two hidden layer configurations , early stopping enabled

**`train_models()`** iterates over all four models. Each goes through `RandomizedSearchCV` with 5-fold stratified cross-validation scoring on F1. The best configuration is refitted on the full training set and evaluated on the held-out test set. Using the same pipeline for every model ensures a fair comparison.

In [7]:
# =============================================================================
# STAGE 5: MODEL TRAINING
# =============================================================================
def build_models():
    return {
        'Logistic Regression': (
            LogisticRegression(max_iter=1000, class_weight='balanced',
                               random_state=Config.RANDOM_STATE),
            {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2']}
        ),
        'Random Forest': (
            RandomForestClassifier(class_weight='balanced',
                                   random_state=Config.RANDOM_STATE, n_jobs=-1),
            {'n_estimators': [100, 200], 'max_depth': [8, 12, None],
             'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}
        ),
        'Gradient Boosting': (
            GradientBoostingClassifier(random_state=Config.RANDOM_STATE),
            {'n_estimators': [100, 200], 'max_depth': [3, 5],
             'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]}
        ),
        'Neural Network': (
            MLPClassifier(random_state=Config.RANDOM_STATE,
                          early_stopping=True, validation_fraction=0.15),
            {'hidden_layer_sizes': [(64, 32), (128, 64)],
             'learning_rate_init': [0.001, 0.01], 'alpha': [0.0001, 0.001]}
        ),
    }
 
 
def train_models(models, X_train, y_train, X_test, y_test):
    header("STAGE 5: MODEL TRAINING")
 
    results = {}
    cv = StratifiedKFold(n_splits=Config.CV_FOLDS, shuffle=True,
                         random_state=Config.RANDOM_STATE)
 
    for name, (model, params) in models.items():
        subheader(f"Training: {name}")
        start = time.time()
 
        n_iter = min(Config.HYPERPARAM_ITER,
                     np.prod([len(v) for v in params.values()]))
 
        search = RandomizedSearchCV(
            model, params, n_iter=n_iter, cv=cv, scoring='f1',
            random_state=Config.RANDOM_STATE, n_jobs=-1, refit=True)
        search.fit(X_train, y_train)
 
        best = search.best_estimator_
        y_pred = best.predict(X_test)
        y_proba = best.predict_proba(X_test)[:, 1]
 
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, y_proba),
        }
 
        results[name] = {
            'model': best, 'y_pred': y_pred, 'y_proba': y_proba,
            'metrics': metrics, 'best_params': search.best_params_,
        }
 
        elapsed = time.time() - start
        print(f"  Best params: {search.best_params_}")
        print(f"  F1: {metrics['f1']:.3f}  ROC-AUC: {metrics['roc_auc']:.3f}  "
              f"Accuracy: {metrics['accuracy']:.3f}  ({elapsed:.1f}s)")
 
    return results
 

## Stage 6 — Evaluation and Comparisons
Three evaluation components:

**`evaluate()`** generates four figures for the best model:
- Figure 6: Bar chart comparing F1 and ROC-AUC across all models
- Figure 7: ROC curves showing discriminative ability
- Figure 8: Precision-recall curves (more informative than ROC for imbalanced data)
- Figure 9: Confusion matrix for the best model (Random Forest)

**`tier_comparison()`** runs the controlled Tier 1 vs Tier 2 experiment. Both use Gradient Boosting with identical settings; only the feature set changes.

**`smote_comparison()`** trains Gradient Boosting with and without SMOTE on the same data. 

In [8]:
# =============================================================================
# STAGE 6: EVALUATION AND VISUALISATION
# =============================================================================
def evaluate(results, y_test):
    header("STAGE 6: EVALUATION")
 
    names = list(results.keys())
    short = [n.replace(' ', '\n') for n in names]
 
    # --- Figure 6: Model comparison ---
    subheader("Figure 6: Model Comparison")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    x = np.arange(len(names))
 
    f1s = [results[n]['metrics']['f1'] for n in names]
    axes[0].bar(x, f1s, color=Config.COLORS[:len(names)], edgecolor='white', width=0.6)
    axes[0].set_xticks(x); axes[0].set_xticklabels(short, fontsize=9)
    axes[0].set_ylabel('F1 Score (Viral Class)')
    axes[0].set_title('F1 Score Comparison')
    axes[0].set_ylim(0, 1.05)
    for i, v in enumerate(f1s):
        axes[0].text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
 
    aucs = [results[n]['metrics']['roc_auc'] for n in names]
    axes[1].bar(x, aucs, color=Config.COLORS[:len(names)], edgecolor='white', width=0.6)
    axes[1].set_xticks(x); axes[1].set_xticklabels(short, fontsize=9)
    axes[1].set_ylabel('ROC AUC')
    axes[1].set_title('ROC AUC Comparison')
    axes[1].set_ylim(0, 1.05)
    for i, v in enumerate(aucs):
        axes[1].text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, '06_model_comparison')
 
    # --- Figure 7: ROC curves ---
    subheader("Figure 7: ROC Curves")
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, name in enumerate(names):
        fpr, tpr, _ = roc_curve(y_test, results[name]['y_proba'])
        ax.plot(fpr, tpr, color=Config.COLORS[i], lw=2,
                label=f"{name} ({results[name]['metrics']['roc_auc']:.3f})")
    ax.plot([0,1],[0,1], 'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves')
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    save_fig(fig, '07_roc_curves')
 
    # --- Figure 8: Precision-Recall curves ---
    subheader("Figure 8: Precision Recall Curves")
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, name in enumerate(names):
        prec, rec, _ = precision_recall_curve(y_test, results[name]['y_proba'])
        pr_auc = auc(rec, prec)
        ax.plot(rec, prec, color=Config.COLORS[i], lw=2,
                label=f"{name} ({pr_auc:.3f})")
    baseline = y_test.sum() / len(y_test)
    ax.axhline(baseline, color='grey', ls=':', alpha=0.6, label=f'Baseline ({baseline:.2f})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title('Precision Recall Curves')
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, '08_precision_recall')
 
    # --- Figure 9: Confusion matrix (best model) ---
    best_name = max(results, key=lambda k: results[k]['metrics']['f1'])
    best = results[best_name]
 
    subheader(f"Figure 9: Confusion Matrix ({best_name})")
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(y_test, best['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Not Viral', 'Viral'],
                yticklabels=['Not Viral', 'Viral'], ax=ax, cbar=False,
                annot_kws={'size': 14, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix: {best_name}')
    plt.tight_layout()
    save_fig(fig, '09_confusion_matrix')
 
    # --- Classification report ---
    subheader(f"Classification Report ({best_name})")
    print(classification_report(y_test, best['y_pred'],
                                target_names=['Not Viral', 'Viral']))
 
    # --- Results summary ---
    subheader("Results Summary")
    summary = pd.DataFrame([
        {'Model': n, **results[n]['metrics']} for n in names
    ])
    print(summary.to_string(index=False, float_format='{:.3f}'.format))
 
    return best_name, best
 
 
def tier_comparison(df, tier1_features, tier2_features):
    """Compare Tier 1 (metadata only) vs Tier 2 (metadata + engagement)."""
    subheader("Tier Comparison")
 
    y = df['viral'].values
    tier_results = {}
 
    for label, features in [('Tier 1 (Metadata)', tier1_features),
                             ('Tier 2 (+ Engagement)', tier2_features)]:
        X = df[features].fillna(0).values
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=Config.TEST_SIZE,
            random_state=Config.RANDOM_STATE, stratify=y)
 
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
 
        X_tr_sm, y_tr_sm = apply_smote(X_tr, y_tr, Config.RANDOM_STATE)
 
        model = GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1,
            random_state=Config.RANDOM_STATE)
        model.fit(X_tr_sm, y_tr_sm)
 
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]
 
        tier_results[label] = {
            'f1': f1_score(y_te, y_pred),
            'roc_auc': roc_auc_score(y_te, y_proba),
        }
        print(f"  {label}: F1={tier_results[label]['f1']:.3f}, "
              f"ROC-AUC={tier_results[label]['roc_auc']:.3f}")
 
    # Plot
    fig, ax = plt.subplots(figsize=(6, 4))
    labels = list(tier_results.keys())
    f1s = [tier_results[l]['f1'] for l in labels]
    aucs = [tier_results[l]['roc_auc'] for l in labels]
    x = np.arange(2); w = 0.3
    ax.bar(x-w/2, f1s, w, label='F1', color=Config.COLORS[0], edgecolor='white')
    ax.bar(x+w/2, aucs, w, label='ROC-AUC', color=Config.COLORS[1], edgecolor='white')
    for i in range(2):
        ax.text(i-w/2, f1s[i]+0.02, f'{f1s[i]:.3f}', ha='center', fontsize=9, fontweight='bold')
        ax.text(i+w/2, aucs[i]+0.02, f'{aucs[i]:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([l.replace(' ', '\n') for l in labels])
    ax.set_ylabel('Score'); ax.set_title('Feature Tier Comparison')
    ax.legend(); ax.set_ylim(0, 1.15)
    plt.tight_layout()
    save_fig(fig, '10_tier_comparison')
 
    return tier_results
 
 
def smote_comparison(X_train, y_train, X_test, y_test):
    """Compare performance with and without SMOTE."""
    subheader("SMOTE Impact")
 
    smote_results = {}
    for label, X_tr, y_tr in [
        ('Without SMOTE', X_train, y_train),
        ('With SMOTE', *apply_smote(X_train, y_train, Config.RANDOM_STATE))
    ]:
        model = GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1,
            random_state=Config.RANDOM_STATE)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_test)
        smote_results[label] = f1_score(y_test, y_pred)
        print(f"  {label}: F1 = {smote_results[label]:.3f}")
 
    fig, ax = plt.subplots(figsize=(6, 4))
    labels = list(smote_results.keys()); vals = list(smote_results.values())
    ax.bar(labels, vals, color=[Config.COLORS[4], Config.COLORS[1]],
           edgecolor='white', width=0.5)
    for i, v in enumerate(vals):
        ax.text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylabel('F1 Score (Viral Class)')
    ax.set_title('Impact of SMOTE')
    ax.set_ylim(0, 1.05)
    delta = vals[1] - vals[0]
    ax.annotate(f'Delta = {delta:+.3f}', xy=(0.5, max(vals)*0.85),
                fontsize=12, fontweight='bold', ha='center', color=Config.COLORS[1])
    plt.tight_layout()
    save_fig(fig, '11_smote_impact')
 
    return smote_results
 

## Stage 7 — Feature Importance and Interpretability
Computes two complementary importance measures:

- **Built-in importance** (`feature_importances_`): Measures how much each feature reduces impurity across all tree splits. Specific to tree-based models.
- **Permutation importance** (`permutation_importance`): Model-agnostic. Shuffles one feature at a time over 15 repeats and measures the F1 drop. If shuffling a feature barely changes performance, it is not important.

Both methods are plotted side by side (Figure 12). 
**Dependence plots** (Figure 13) show the relationship between each of the top 3 features and the predicted viral probability. A degree-3 polynomial is fitted using `np.polyfit` to show the trend. 

In [9]:
# =============================================================================
# STAGE 7: FEATURE IMPORTANCE
# =============================================================================
def feature_importance(best_model, X_test, y_test, feature_names):
    header("STAGE 7: FEATURE IMPORTANCE")
 
    # Built-in importance
    fi_df = None
    if hasattr(best_model, 'feature_importances_'):
        fi = best_model.feature_importances_
        fi_df = pd.DataFrame({'feature': feature_names, 'importance': fi}
                             ).sort_values('importance', ascending=True)
        subheader("Built-in Feature Importance")
        print(fi_df.to_string(index=False))
 
    # Permutation importance
    subheader("Permutation Importance")
    perm = permutation_importance(
        best_model, X_test, y_test, n_repeats=15,
        random_state=Config.RANDOM_STATE, n_jobs=-1, scoring='f1')
    pi_df = pd.DataFrame({
        'feature': feature_names,
        'importance': perm.importances_mean,
        'std': perm.importances_std,
    }).sort_values('importance', ascending=True)
    print(pi_df.to_string(index=False))
 
    # --- Figure 12: Feature importance ---
    subheader("Figure 12: Feature Importance")
    if fi_df is not None:
        fig, axes = plt.subplots(1, 2, figsize=(11, 5))
        axes[0].barh(fi_df['feature'], fi_df['importance'],
                     color=Config.COLORS[0], alpha=0.85)
        axes[0].set_xlabel('Impurity Based Importance')
        axes[0].set_title('Built-in Importance')
        axes[1].barh(pi_df['feature'], pi_df['importance'],
                     xerr=pi_df['std'], color=Config.COLORS[1], alpha=0.85, capsize=3)
        axes[1].set_xlabel('Mean F1 Decrease')
        axes[1].set_title('Permutation Importance')
        plt.suptitle('Feature Importance: Gradient Boosting',
                     fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        save_fig(fig, '12_feature_importance')
    else:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh(pi_df['feature'], pi_df['importance'],
                xerr=pi_df['std'], color=Config.COLORS[1], alpha=0.85, capsize=3)
        ax.set_xlabel('Mean F1 Decrease')
        ax.set_title('Permutation Feature Importance')
        plt.tight_layout()
        save_fig(fig, '12_feature_importance')
 
    # --- Figure 13: Dependence plots ---
    subheader("Figure 13: Dependence Plots")
    top3 = pi_df.tail(3)['feature'].tolist()[::-1]
    y_proba = best_model.predict_proba(X_test)[:, 1]
 
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for i, feat in enumerate(top3):
        idx = feature_names.index(feat)
        vals = X_test[:, idx]
        axes[i].scatter(vals, y_proba, c=y_proba, cmap='RdYlBu_r',
                        alpha=0.3, s=8, edgecolors='none')
        z = np.polyfit(vals, y_proba, 3)
        xs = np.linspace(vals.min(), vals.max(), 100)
        axes[i].plot(xs, np.clip(np.poly1d(z)(xs), 0, 1),
                     color=Config.COLORS[1], lw=2)
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel('P(Viral)' if i == 0 else '')
        axes[i].set_title(feat)
    plt.suptitle('Feature Dependence (Top 3)', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, '13_dependence_plots')
 

## Stage 8 — Cross-Validation Stability
Tests whether the results performs consistently across different splits of the data. Uses 5-fold stratified cross-validation on the SMOTE-balanced training set, computing F1, ROC-AUC, and accuracy for each fold.

In [10]:
# =============================================================================
# STAGE 8: CROSS-VALIDATION STABILITY
# =============================================================================
def cv_stability(X_train, y_train):
    header("STAGE 8: CROSS-VALIDATION STABILITY")
 
    model = GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=Config.RANDOM_STATE)
 
    cv = StratifiedKFold(n_splits=Config.CV_FOLDS, shuffle=True,
                         random_state=Config.RANDOM_STATE)
 
    cv_results = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring={'f1': make_scorer(f1_score), 'roc_auc': 'roc_auc', 'accuracy': 'accuracy'},
        n_jobs=-1)
 
    for metric in ['f1', 'roc_auc', 'accuracy']:
        scores = cv_results[f'test_{metric}']
        print(f"  {metric:>10s}: {scores.mean():.3f} +/- {scores.std():.3f}  "
              f"[{', '.join(f'{s:.3f}' for s in scores)}]")
 
    fig, ax = plt.subplots(figsize=(7, 4))
    cv_data = pd.DataFrame({
        'F1': cv_results['test_f1'],
        'ROC AUC': cv_results['test_roc_auc'],
        'Accuracy': cv_results['test_accuracy'],
    })
    bp = cv_data.boxplot(ax=ax, patch_artist=True, return_type='dict')
    for patch, color in zip(bp['boxes'], Config.COLORS[:3]):
        patch.set_facecolor(color); patch.set_alpha(0.6)
    ax.set_ylabel('Score')
    ax.set_title(f'Cross-Validation Stability ({Config.CV_FOLDS}-Fold)')
    plt.tight_layout()
    save_fig(fig, '14_cv_stability')
 
    return cv_results

## Run the Full Pipeline
The `main()` function executes all stages in order:
1. **Setup** — Create output directories
2. **Load data** — CSV loading, deduplication, virality labelling
3. **EDA** — Generate 5 exploratory figures
4. **Feature engineering** — Cyclical encoding, log transforms, tier definitions
5. **Split and SMOTE** — 80/20 stratified split, then SMOTE on training set only
6. **Train** — 4 models with RandomizedSearchCV
7. **Evaluate** — Model comparison, tier comparison, SMOTE comparison
8. **Feature importance** — Built-in and permutation importance, dependence plots
9. **CV stability** — 5-fold cross-validation

Output: 14 figures saved to `./output/figures/`, processed data and model results saved as CSV files.

In [11]:
# =============================================================================
# MAIN
# =============================================================================
def main():
    start_time = time.time()
 
    print("="*70)
    print("  PREDICTING VIDEO VIRALITY ON YOUTUBE")
    print("  Kaggle Dataset: datasnaek/youtube-new")
    print("="*70)
 
    setup()
 
    # Stage 1: Load data
    df = load_data()
 
    # Stage 2: EDA
    df = run_eda(df)
 
    # Stage 3: Feature engineering
    df, tier1_features, tier2_features = engineer_features(df)
 
    # Stage 4: Split and SMOTE
    header("STAGE 4: TRAIN-TEST SPLIT AND SMOTE")
 
    y = df['viral'].values
    X = df[tier2_features].fillna(0).values
 
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=Config.TEST_SIZE,
        random_state=Config.RANDOM_STATE, stratify=y)
 
    print(f"  Train: {len(y_train)} ({(y_train==1).sum()} viral)")
    print(f"  Test:  {len(y_test)} ({(y_test==1).sum()} viral)")
 
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)
 
    X_train_smote, y_train_smote = apply_smote(
        X_train_scaled, y_train, Config.RANDOM_STATE)
 
    # Stage 5: Train models
    models = build_models()
    results = train_models(
        models, X_train_smote, y_train_smote, X_test_scaled, y_test)
 
    # Stage 6: Evaluate
    best_name, best = evaluate(results, y_test)
    print(f"\n  Best model: {best_name}")
 
    tier_results = tier_comparison(df, tier1_features, tier2_features)
    smote_results = smote_comparison(X_train_scaled, y_train, X_test_scaled, y_test)
 
    # Stage 7: Feature importance
    fi = feature_importance(
        best['model'], X_test_scaled, y_test, tier2_features)
 
    # Stage 8: CV stability
    cv_results = cv_stability(X_train_smote, y_train_smote)
 
    # Final summary
    elapsed = time.time() - start_time
    header("PIPELINE COMPLETE")
    print(f"  Runtime:     {elapsed:.1f}s")
    print(f"  Best model:  {best_name}")
    print(f"  F1:          {best['metrics']['f1']:.3f}")
    print(f"  ROC-AUC:     {best['metrics']['roc_auc']:.3f}")
    print(f"  Accuracy:    {best['metrics']['accuracy']:.3f}")
    print(f"  CV F1 std:   {cv_results['test_f1'].std():.3f}")
    print(f"  Figures:     {Config.FIGURE_DIR}")
 
    # Save results
    summary = pd.DataFrame([
        {'Model': n, **results[n]['metrics'],
         'Params': str(results[n]['best_params'])} for n in results
    ])
    summary.to_csv(os.path.join(Config.OUTPUT_DIR, 'model_results.csv'), index=False)
    print(f"  Results:     {Config.OUTPUT_DIR}model_results.csv")
 
    return results, best_name
 
 
if __name__ == '__main__':
    results, best_name = main()

  PREDICTING VIDEO VIRALITY ON YOUTUBE
  Kaggle Dataset: datasnaek/youtube-new

  STAGE 1: DATA LOADING AND PREPROCESSING
  Found 3 CSV files: ['CAvideos.csv', 'GBvideos.csv', 'USvideos.csv']
  Loaded CAvideos.csv: 40881 rows, 17 categories
  Loaded GBvideos.csv: 38916 rows, 16 categories
  Loaded USvideos.csv: 40949 rows, 16 categories

  Combined: 120746 rows
  After deduplication: 34050 rows
  After cleaning: 34025 rows

--- Computing metadata features ---

--- Computing title sentiment ---
  Using VADER sentiment analyzer

--- Computing virality labels ---
  Viral videos: 5111 (15.0%)
  Non-viral:    28914 (85.0%)
  Categories:   18

  Saved processed data to ./output/processed_data.csv

  STAGE 2: EXPLORATORY DATA ANALYSIS

--- Figure 1: Class Distribution ---
  Saved: ./output/figures/01_class_distribution.png

--- Figure 2: Category Analysis ---
  Saved: ./output/figures/02_categories.png

--- Figure 3: Correlation Matrix ---
  Saved: ./output/figures/03_correlation.png

--- Fig